In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze


In [2]:


spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

your 131072x1 screen size is bogus. expect trouble
26/05/04 21:48:54 WARN Utils: Your hostname, DESKTOP-S3JB25M resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/04 21:48:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/felipe/.ivy2/cache
The jars for the packages stored in: /home/felipe/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8dc260f-3ba7-48fa-8eb5-42ce4c3d1775;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 340ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hado

SparkSession criada com sucesso!


In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] ja existe
Buckets: ['bronze', 'landing-zone']


In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

5 arquivos CSV encontrados no bucket [landing-zone]:
  - albuns.csv
  - artistas.csv
  - musicas.csv
  - reproducoes.csv
  - usuarios.csv


In [5]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)

    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)

    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

Convertendo 5 CSVs para Delta Lake...



26/05/04 21:49:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/04 21:49:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  albuns: 5 registros | 5 colunas -> s3a://bronze/albuns
  artistas: 5 registros | 5 colunas -> s3a://bronze/artistas


  musicas: 5 registros | 5 colunas -> s3a://bronze/musicas
  reproducoes: 5 registros | 5 colunas -> s3a://bronze/reproducoes
  usuarios: 5 registros | 5 colunas -> s3a://bronze/usuarios

Conversao concluida! 5 tabelas Delta criadas no bucket [bronze].


In [6]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)

    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake...



  albuns: Delta=True | 5 registros | Colunas: ['id', 'artista_id', 'titulo', 'ano_lancamento', 'total_faixas']


  artistas: Delta=True | 5 registros | Colunas: ['id', 'nome', 'pais', 'genero', 'criado_em']
  musicas: Delta=True | 5 registros | Colunas: ['id', 'album_id', 'titulo', 'duracao_segundos', 'numero_faixa']
  reproducoes: Delta=True | 5 registros | Colunas: ['id', 'usuario_id', 'musica_id', 'reproduzida_em', 'duracao_ouvida_segundos']
  usuarios: Delta=True | 5 registros | Colunas: ['id', 'nome', 'email', 'plano', 'criado_em']


In [7]:
response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET, Delimiter='/')
tabelas = [p['Prefix'].rstrip('/') for p in response.get('CommonPrefixes', [])]

for tabela in tabelas:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- ALBUNS ---
+---+----------+-------------------+--------------+------------+
| id|artista_id|             titulo|ano_lancamento|total_faixas|
+---+----------+-------------------+--------------+------------+
|  1|         1|        After Hours|          2020|          14|
|  2|         2|   Future Nostalgia|          2020|          11|
|  3|         3|To Pimp a Butterfly|          2015|          16|
|  4|         4|  Happier Than Ever|          2021|          16|
|  5|         5|           Currents|          2015|          13|
+---+----------+-------------------+--------------+------------+


--- ARTISTAS ---
+---+--------------+--------------+-----------+--------------------+
| id|          nome|          pais|     genero|           criado_em|
+---+--------------+--------------+-----------+--------------------+
|  1|    The Weeknd|        Canadá|        R&B|2026-05-05 00:26:...|
|  2|      Dua Lipa|   Reino Unido|        Pop|2026-05-05 00:26:...|
|  3|Kendrick Lamar|Estados Unidos|

+---+-------------+--------------------+-------+--------------------+
| id|         nome|               email|  plano|           criado_em|
+---+-------------+--------------------+-------+--------------------+
|  1|     Ana Lima|  ana.lima@email.com|premium|2026-05-05 00:26:...|
|  2|  Bruno Costa|bruno.costa@email...|   free|2026-05-05 00:26:...|
|  3|  Carla Souza|carla.souza@email...|premium|2026-05-05 00:26:...|
|  4|Diego Martins|diego.martins@ema...|   free|2026-05-05 00:26:...|
|  5| Elena Ferraz|elena.ferraz@emai...|premium|2026-05-05 00:26:...|
+---+-------------+--------------------+-------+--------------------+



In [8]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
